# 3c. COMPASS PreTrainer Embedding Extraction (No Data Leakage)

Извлечение geneset-level эмбеддингов из **базовой** COMPASS модели (`pretrainer.pt`).

**Почему PreTrainer?**
- `finetuner_pft_all.pt` был fine-tuned на 1,133 ICI пациентах, включая
  когорты IMmotion150, IMVigor210 и др. — **data leakage** при оценке.
- `pretrainer.pt` обучен self-supervised только на TCGA экспрессии →
  **нет утечки ICI данных**, одинаковый для всех когорт.

**Архитектура** та же:
```
Gene TPM → TransformerEncoder → GenesetProjector (132 × 32) → 4,224-dim
```

**Данные:**
- Исходные когорты загружаются из **`data/raw`**
- Результаты PreTrainer сохраняются в **`data/interim_PT`**

## 0. Environment Setup

In [ ]:
# !rm -rf batchcor-rna-embeds/  # if already cloned
!git clone https://github.com/onion-42/batchcor-rna-embeds.git 2>/dev/null || (cd batchcor-rna-embeds && git pull origin main)
%cd batchcor-rna-embeds

!pip install -q uv
!uv pip install --system -e .
!pip install -q --upgrade ipython ipykernel

In [ ]:
import os
import sys
import shutil
from pathlib import Path

import numpy as np
import torch
from loguru import logger

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    BATCH_COL,
    DIAGNOSIS_COL,
    COHORT_CANCER_CODE,
    SEED,
    COMPASS_N_PCA,
    COMPASS_N_UMAP,
    # PreTrainer-specific keys (no ICI fine-tuning)
    COMPASS_PT_MODEL_URL,
    COMPASS_PT_EMBEDDING_KEY,
    COMPASS_PT_PCA_KEY,
    COMPASS_PT_UMAP_KEY,
    COMPASS_PT_METADATA_KEY,
)
from batchcor_rna_emb.data_io import load_cohort, save_adata_zarr
from batchcor_rna_emb.compass_embedder import run_compass_pipeline

set_logger(level="INFO")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info("Device: {}", DEVICE)
logger.info("PyTorch: {}, CUDA available: {}", torch.__version__, torch.cuda.is_available())

## 1. Mount Drive & Define Paths

- **Source**: `data/raw` — исходные когорты
- **Destination**: `data/interim_PT` — результаты PreTrainer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Source: raw data
DATA_RAW_DIR = Path('/content/drive/MyDrive/data/raw')
# Destination: separate folder for PreTrainer results
DATA_INTERIM_PT_DIR = Path('/content/drive/MyDrive/data/interim_PT')
DATA_INTERIM_PT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path('/content/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

logger.info("DATA_RAW_DIR: {}", DATA_RAW_DIR)
logger.info("DATA_INTERIM_PT_DIR: {}", DATA_INTERIM_PT_DIR)
logger.info("Available cohorts (raw): {}", [p.name for p in sorted(DATA_RAW_DIR.glob('*.adata.zarr'))])

## 2. Download COMPASS PreTrainer Model

In [ ]:
PT_MODEL_PATH = MODEL_DIR / "pretrainer.pt"

if not PT_MODEL_PATH.exists():
    logger.info("Downloading COMPASS PreTrainer from {}", COMPASS_PT_MODEL_URL)
    !wget -q -O {PT_MODEL_PATH} {COMPASS_PT_MODEL_URL}
    logger.info("Model downloaded: {} ({:.1f} MB)", PT_MODEL_PATH, PT_MODEL_PATH.stat().st_size / 1e6)
else:
    logger.info("Model already exists: {}", PT_MODEL_PATH)

## 3. Process All Cohorts with PreTrainer

Для каждой когорты:
1. Load from `data/raw`
2. Extract PreTrainer geneset-level embeddings → `FM_COMPASS_PT_embedding`
3. Compute PCA-128D → `PCA128d_FM_COMPASS_PT_embedding`
4. Compute UMAP-3D → `UMAP3d_FM_COMPASS_PT_embedding`
5. Store metadata → `FM_COMPASS_PT_metadata`
6. Save to `data/interim_PT`

In [ ]:
BATCH_SIZE = 128

for cohort_name, cancer_code in COHORT_CANCER_CODE.items():
    zarr_path    = DATA_RAW_DIR / f"{cohort_name}.adata.zarr"
    interim_path = DATA_INTERIM_PT_DIR / f"{cohort_name}.adata.zarr"

    if not zarr_path.exists():
        logger.warning("Cohort not found in raw, skipping: {}", zarr_path)
        continue

    if interim_path.exists():
        logger.info("Already in interim_PT, skipping: {}", cohort_name)
        continue

    logger.info("\n{'=' * 70}")
    logger.info("Processing cohort (PreTrainer): {}", cohort_name)
    logger.info("{'=' * 70}")

    # Load from raw
    adata = load_cohort(zarr_path)
    logger.info("Loaded: {} samples × {} genes", adata.n_obs, adata.n_vars)
    logger.info("Batch distribution:\n{}", adata.obs[BATCH_COL].value_counts())
    logger.info("Diagnosis distribution:\n{}", adata.obs[DIAGNOSIS_COL].value_counts())

    # Run COMPASS pipeline with PreTrainer model + PT-specific keys
    adata = run_compass_pipeline(
        adata,
        model_path=PT_MODEL_PATH,
        cancer_code=cancer_code,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        seed=SEED,
        # PreTrainer-specific keys
        embedding_key=COMPASS_PT_EMBEDDING_KEY,
        pca_key=COMPASS_PT_PCA_KEY,
        umap_key=COMPASS_PT_UMAP_KEY,
        metadata_key=COMPASS_PT_METADATA_KEY,
    )

    # Verify
    logger.info("Verification:")
    logger.info("  .obsm['{}'] shape: {}, dtype: {}",
        COMPASS_PT_EMBEDDING_KEY,
        adata.obsm[COMPASS_PT_EMBEDDING_KEY].shape,
        adata.obsm[COMPASS_PT_EMBEDDING_KEY].dtype,
    )
    logger.info("  .obsm['{}'] shape: {}",
        COMPASS_PT_PCA_KEY, adata.obsm[COMPASS_PT_PCA_KEY].shape,
    )
    logger.info("  .obsm['{}'] shape: {}",
        COMPASS_PT_UMAP_KEY, adata.obsm[COMPASS_PT_UMAP_KEY].shape,
    )
    logger.info("  .uns['{}']: {}",
        COMPASS_PT_METADATA_KEY, adata.uns[COMPASS_PT_METADATA_KEY],
    )

    # Save to interim_PT
    save_adata_zarr(adata, interim_path)
    logger.info("Saved to interim_PT: {}", interim_path)

    # Free memory
    del adata
    torch.cuda.empty_cache() if DEVICE == "cuda" else None

logger.info("\nAll cohorts processed with PreTrainer.")
logger.info("interim_PT contents: {}", [p.name for p in sorted(DATA_INTERIM_PT_DIR.glob('*.adata.zarr'))])

## 4. Verification — Spot Check

In [ ]:
test_cohort = "NSCLC_Tissue_ICI_Pred"
test_path = DATA_INTERIM_PT_DIR / f"{test_cohort}.adata.zarr"

if test_path.exists():
    adata_check = load_cohort(test_path)

    assert COMPASS_PT_EMBEDDING_KEY in adata_check.obsm, "Missing PT embedding!"
    assert COMPASS_PT_PCA_KEY in adata_check.obsm, "Missing PT PCA!"
    assert COMPASS_PT_UMAP_KEY in adata_check.obsm, "Missing PT UMAP!"
    assert COMPASS_PT_METADATA_KEY in adata_check.uns, "Missing PT metadata!"

    emb = adata_check.obsm[COMPASS_PT_EMBEDDING_KEY]
    assert emb.dtype == np.float32, f"Wrong dtype: {emb.dtype}"
    assert emb.shape[0] == adata_check.n_obs, "Row count mismatch!"

    logger.info("All assertions passed for '{}' (PreTrainer).", test_cohort)
    logger.info("  PT Embedding shape: {}", emb.shape)
    logger.info("  PT PCA shape: {}", adata_check.obsm[COMPASS_PT_PCA_KEY].shape)
    logger.info("  PT UMAP shape: {}", adata_check.obsm[COMPASS_PT_UMAP_KEY].shape)
    logger.info("  PT Metadata: {}", adata_check.uns[COMPASS_PT_METADATA_KEY])
else:
    logger.warning("Test cohort not found in interim_PT: {}", test_path)